## what we are going to do is scale everything up

In [2]:
#the imports
#cell that have all of the imports
# Data
import numpy as np
import pandas as pd

# visualization
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# pytorch
import torch
import torch.nn.functional as F
from torch import nn, tensor
from torch.utils.data import DataLoader, Dataset

# torchvision
from torchvision import transforms, models
from torchvision.transforms import functional as TF

# fastai
from fastai.vision.all import *
from fastai.tabular.all import *
from fastai.data.transforms import RandomSplitter
from fastai.callback.hook import *

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

/home/seif/Desktop/PycharmProjects/AI&ML dbrouke course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/seif/Desktop/PycharmProjects/AI&ML dbrouke course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## one of the main problems that we can have when we are scalinng --> memory

In [4]:
import os
os.getcwd()

'/kaggle/working'

In [5]:
# first here we are going to get the smallest train class in the dataset to see will that memory crash when we load the model (smallest class is to minimize the epoch time)
path = Path("/kaggle/input/competitions/paddy-disease-classification")
df = pd.read_csv(path/"train.csv")
df.label.value_counts()

label
normal                      1764
blast                       1738
hispa                       1594
dead_heart                  1442
tungro                      1088
brown_spot                   965
downy_mildew                 620
bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
Name: count, dtype: int64

 so we are going to use this --> bacterial_panicle_blight


In [6]:
trn_path = path/'train_images'/'bacterial_panicle_blight'

In [7]:
tst_files = get_image_files(path/"test_images").sorted()

fine_tune() : freeze body, train head only → unfreeze all, train everything

fit_one_cycle() : train everything ( if i make a manual unfreeze before) , train only the classifier (if i dont make a manual freeze)

In [8]:
# now lets make a training function
def train (model_name, #arch
           size_image, # image size to train on
           train_path = trn_path,
           item_tfms = Resize(480,method='squish'), # resize images
           accum = 1, # this is new technique which is explained in the markdown below
           fine_tune = True, # full or one_cycle ?
           epochs = 12 ,
           lr = 0.1):

    dls = ImageDataLoaders.from_folder(train_path,
                                       valid_pct=0.2,
                                       item_tfms = item_tfms, # this is applied to every image before batching
                                       batch_tfms= aug_transforms(size=size_image,min_scale=0.75),# make the augmentation for the images in batches during training and it insures that atleast 75% (0.75) of the image is cropped
                                       bs = 64//accum)

    # now lets make the gradient accumulation call (callbacks == cbs)
    cbs = GradientAccumulation(64) if accum else [] # this make it apply the GradientAccumulation if there was value from us , else --> normal full batch training

    # now lets make the learner
    learn = vision_learner(dls,
                           model_name,
                           metrics=[accuracy,error_rate],
                           cbs=cbs).to_fp16()

    ### now let make the training
    # fine_tune (real_training)
    if fine_tune:
        # train
        learn.fine_tune(epochs,lr)
        # return the tta preds
        return learn.tta(dl=dls.test_dl(tst_files)) # returning a tta for the test_dl that was made by teh same setting as the train dls but removing teh batch_tfms , val_pct , shuffle , etc (things that aren't in test
    else:
        # make a memory test (this is made by default in the fine_tune,but not here)
        learn.unfreeze()
        #make the one cycle and return nothing because this is just a memory test (know if it crashes)
        learn.fit_one_cycle(epochs,lr)




    PROBLEM:
    big model + big images = GPU out of memory ❌

    NAIVE FIX: just use smaller batch
    batch=8 instead of batch=64 → fits in memory ✓
    BUT small batches = noisy gradients = worse training ✗

    SMART FIX: gradient accumulation
    batch=8 × 8 times = same as batch=64
    fits in memory AND good gradients ✓

    WITHOUT accum:                    WITH accum=8:
    ┌────────┐                        ┌────────┐
    │8 images│→ update weights        │8 images│→ save gradients
    └────────┘                        └────────┘
    ┌────────┐                        ┌────────┐
    │8 images│→ update weights        │8 images│→ ADD gradients
    └────────┘                        └────────┘
    ┌────────┐                        ┌────────┐
    │8 images│→ update weights        │8 images│→ ADD gradients
    └────────┘                        └────────┘
    ┌────────┐                        ┌────────┐
    │8 images│→ update weights        │8 images│→ ADD gradients
    └────────┘                        └────────┘
        ↑                             ┌────────┐
    8 noisy updates                   │8 images│→ ADD gradients
                                      └────────┘
                                      ┌────────┐
                                      │8 images│→ ADD gradients
                                      └────────┘
                                      ┌────────┐
                                      │8 images│→ ADD gradients
                                      └────────┘
                                      ┌────────┐
                                      │8 images│→ ADD → UPDATE ✓
                                      └────────┘
                                          ↑
                                   1 clean update
                                   same as 64 images



        #accum=1 → normal training, batch=64
        # accum=2 → batch=32, accumulate 2 batches before update
        # accum=4 → batch=16, accumulate 4 batches before update

In [9]:
# now we are going to try the training funcion
train("convnext_small_in22k",128,epochs = 2,accum = 1,fine_tune = False)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_small_in22k to current convnext_small.fb_in22k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/265M [00:00<?, ?B/s]

epoch,train_loss,valid_loss,accuracy,error_rate,time
0,0.000000,0.000000,1.000000,0.000000,00:18
1,0.000000,0.000000,1.000000,0.000000,00:02


In [10]:
# here we are going to make a function that print what is in memory and then empty it at the end ( the emptying is for things that arent used  or like overwritten like a old learn that is overwritten)
import gc
def gpu_reporter():
    print(torch.cuda.list_gpu_processes())
    gc.collect()
    torch.cuda.empty_cache()

In [11]:
gpu_reporter()

GPU:0
process         55 uses     4204.000 MB GPU memory


In [12]:
#now lets see what does the gradient accum does ( the memory that is there is like the last batch+activations,etc., but mainly the differnece is the last batch -> how the accum affect batch use in memory
train('convnext_small_in22k', 128, epochs=2, accum=2, fine_tune=False)
gpu_reporter()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_small_in22k to current convnext_small.fb_in22k.
  model = create_fn(


epoch,train_loss,valid_loss,accuracy,error_rate,time
0,0.000000,0.000000,1.000000,0.000000,00:14
1,0.000000,0.000000,1.000000,0.000000,00:02


GPU:0
process         55 uses     2744.000 MB GPU memory


In [13]:
# lets see 4
train('convnext_small_in22k', 128, epochs=2, accum=4, fine_tune=False)
gpu_reporter()

epoch,train_loss,valid_loss,accuracy,error_rate,time
0,0.000000,0.000000,1.000000,0.000000,00:14
1,0.000000,0.000000,1.000000,0.000000,00:02


GPU:0
process         55 uses     1898.000 MB GPU memory


In [14]:
# lets check for a larger model
train('convnext_large_in22k', (320,240), epochs=1, accum=16, fine_tune=False)
gpu_reporter()

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_large_in22k to current convnext_large.fb_in22k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/919M [00:00<?, ?B/s]

epoch,train_loss,valid_loss,accuracy,error_rate,time
0,0.000000,0.000000,1.000000,0.000000,00:21


GPU:0
process         55 uses     5012.000 MB GPU memory


In [15]:
train('vit_large_patch16_224', 224, epochs=1, accum=32, fine_tune=False)
gpu_reporter()

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

epoch,train_loss,valid_loss,accuracy,error_rate,time
0,0.000000,0.000000,1.000000,0.000000,00:15


GPU:0
process         55 uses     5902.000 MB GPU memory


In [16]:
train('efficientnetv2_rw_m', 224, epochs=1, accum=4, fine_tune=False)
gpu_reporter()

model.safetensors:   0%|          | 0.00/214M [00:00<?, ?B/s]

epoch,train_loss,valid_loss,accuracy,error_rate,time
0,0.000000,0.000000,1.000000,0.000000,00:12


GPU:0
process         55 uses     3476.000 MB GPU memory


# now lets train models

In [17]:
# make the original resolution
res = (640,480)

Reason for what we are doing :

 CNN -- needs the pictures to be the perfect landscape where even if we sacrifice some details to make the photo not distorted then make it

  VIT --  it is more pattern recongnizer no matter the landscape

Big Picture: The "How" vs. The "Final Result"
Think of Resize as a specialized tool.

**IT IS EASIER IF YOU LOOK AT THE TRAIN_FN**

The first part (res or 480) tells the tool how to look at the source file. -->

The second part (the number after the comma) tells the tool what size to spit out.

In [18]:
models = {
    'convnext_small_in22k': {
        # Using 128 for very fast iterations/testing
        (Resize(res), 128),
    },
    'convnext_large_in22k': {
        # Using (320, 240) keeps the original 4:3 aspect ratio (becasue if we made the original --> it needs more than 8 GB VRAM) and we need the most detail for one of biggest models here
        # which helps detect long, thin disease streaks on leaves
        (Resize(res), (320, 240)),
    },
    'vit_large_patch16_224': {
        # this is said to make 2 different models one squish and one on the crop
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    },
    'efficientnetv2_rw_m': {
        # EfficientNet is mathematically optimized for 224
        (Resize(res), 224),
    }
}

In [19]:
# make now the full training path images not the small one that was for testing
trn_path = path/"train_images"

In [20]:
for i,j in models.items():
    print(i)
    print("-"*50)
    print(j)
    print("*"*50)


convnext_small_in22k
--------------------------------------------------
{(Resize -- {'size': (480, 640), 'method': 'crop', 'pad_mode': 'reflection', 'resamples': (<Resampling.BILINEAR: 2>, <Resampling.NEAREST: 0>), 'p': 1.0}
(enc:1,dec:0), 128)}
**************************************************
convnext_large_in22k
--------------------------------------------------
{(Resize -- {'size': (480, 640), 'method': 'crop', 'pad_mode': 'reflection', 'resamples': (<Resampling.BILINEAR: 2>, <Resampling.NEAREST: 0>), 'p': 1.0}
(enc:1,dec:0), (320, 240))}
**************************************************
vit_large_patch16_224
--------------------------------------------------
{(Resize -- {'size': (480, 640), 'method': 'crop', 'pad_mode': 'reflection', 'resamples': (<Resampling.BILINEAR: 2>, <Resampling.NEAREST: 0>), 'p': 1.0}
(enc:1,dec:0), 224), (Resize -- {'size': (480, 480), 'method': 'squish', 'pad_mode': 'reflection', 'resamples': (<Resampling.BILINEAR: 2>, <Resampling.NEAREST: 0>), 'p': 1.0

## The training cell

In [21]:
# lets make a list that collects us the final predictions of the trained models [model_1_preds,model_2_preds,....]
tta_result = []
big_accum_models = ["vit_large_patch16_224","convnext_large_in22k"]
### now make the nested loop
for arch,details in models.items(): # we make it .items() to get the keys(arch) and the value(details)
    for item,size in details: # this is make items_tfms , size --> in Training_fn
        print(f"[INFO]  MODEL : {arch}")
        print(f"[INFO]  SIZE : {size}")
        print(f"[INFO]  METHOD : {item.name}")
        if arch in big_accum_models:
            tta_result.append(train(arch,size,train_path=trn_path,item_tfms=item , accum=2,lr=0.01))
        else:
            tta_result.append(train(arch,size,train_path=trn_path,item_tfms=item , accum=1,lr=0.01))

        # make the memory cleaning
        gc.collect()
        torch.cuda.empty_cache()

[INFO]  MODEL : convnext_small_in22k
[INFO]  SIZE : 128
[INFO]  METHOD : Resize -- {'size': (480, 640), 'method': 'crop', 'pad_mode': 'reflection', 'resamples': (<Resampling.BILINEAR: 2>, <Resampling.NEAREST: 0>), 'p': 1.0}



epoch,train_loss,valid_loss,accuracy,error_rate,time
0,1.233577,0.837728,0.736185,0.263815,00:47


epoch,train_loss,valid_loss,accuracy,error_rate,time
0,0.597204,0.369557,0.877463,0.122537,00:46
1,0.485501,0.318111,0.902931,0.097069,00:47
2,0.404214,0.323166,0.898126,0.101874,00:48
3,0.327125,0.237038,0.919270,0.080730,00:47
4,0.273490,0.246926,0.925517,0.074483,00:47
5,0.195121,0.161534,0.955790,0.044210,00:47
6,0.145681,0.148737,0.962038,0.037963,00:47
7,0.114990,0.123204,0.967323,0.032677,00:47
8,0.076653,0.113080,0.971648,0.028352,00:47
9,0.061538,0.100670,0.973090,0.026910,00:47


epoch,train_loss,valid_loss,accuracy,error_rate,time


<div></div>

[INFO]  MODEL : convnext_large_in22k
[INFO]  SIZE : (320, 240)
[INFO]  METHOD : Resize -- {'size': (480, 640), 'method': 'crop', 'pad_mode': 'reflection', 'resamples': (<Resampling.BILINEAR: 2>, <Resampling.NEAREST: 0>), 'p': 1.0}



... (15 outputs left out)

epoch,train_loss,valid_loss,accuracy,error_rate,time
0,2.038876,1.567271,0.505046,0.494954,01:38


epoch,train_loss,valid_loss,accuracy,error_rate,time
0,1.308528,1.088503,0.639596,0.360404,01:57
1,1.047978,0.889509,0.736185,0.263815,01:55
2,0.821446,0.567371,0.814512,0.185488,01:55
3,0.623159,0.403837,0.860644,0.139356,01:55
4,0.447770,0.885529,0.904853,0.095147,01:55
5,0.339328,0.323814,0.917828,0.082172,01:55
6,0.244860,0.216673,0.936088,0.063912,01:55
7,0.172292,0.176136,0.948102,0.051898,01:56
8,0.144063,0.696988,0.946660,0.053340,01:55
9,0.128192,nan,0.947621,0.052379,01:55


epoch,train_loss,valid_loss,accuracy,error_rate,time


<div></div>

In [25]:
# If you are using the fastai helper:
save_pickle('tta_result.pkl', tta_result)

In [26]:
print("Total Models:", len(tta_result))
print("Items in each tuple:", len(tta_result[0]))
print("Shape of Preds in Models :", tta_result[1][0].shape)

Total Models: 5
Items in each tuple: 2
Shape of Preds in Models : torch.Size([3469, 10])


tta_res looks like:

    [(preds_model1, targets1), (preds_model2, targets2), (preds_model3, targets3)]

we are going to extract just the preds , because we are in test set --> targets are just empty


    zip(*tta_res) unpacks rows into columns:

    BEFORE:                      AFTER zip(*...):
    (preds1,   targets1)         (preds1,  preds2,  preds3)   ← first
    (preds2,   targets2)    →    (targets1,targets2,targets3)  ← second
    (preds3,   targets3)

In [34]:
# make a ziped list and take the first index
tta_preds = list(zip(*tta_result))[0]
# preds tuple
tta_preds

(tensor([[1.6659e-08, 3.3985e-09, 4.9637e-09,  ..., 1.0000e+00, 9.0392e-10,
          2.8070e-11],
         [6.2860e-07, 3.9754e-05, 8.1771e-07,  ..., 1.1972e-05, 9.9993e-01,
          1.3862e-06],
         [5.4699e-07, 4.1786e-07, 9.5020e-06,  ..., 2.3758e-04, 3.7607e-04,
          4.2974e-06],
         ...,
         [6.4003e-11, 6.8815e-10, 4.8853e-07,  ..., 6.6333e-10, 1.0000e+00,
          2.6948e-09],
         [1.1145e-05, 9.9641e-01, 1.1117e-05,  ..., 2.0250e-03, 4.5082e-06,
          8.4538e-05],
         [1.7369e-12, 8.1543e-13, 4.4167e-11,  ..., 3.3616e-10, 4.2111e-11,
          4.6603e-10]]),
 tensor([[2.7396e-06, 3.6010e-08, 4.3913e-08,  ..., 9.9967e-01, 9.9969e-07,
          9.4082e-09],
         [1.0126e-06, 2.6014e-06, 1.4784e-07,  ..., 4.4179e-07, 9.9999e-01,
          1.0899e-08],
         [2.0359e-06, 1.3817e-06, 2.3246e-05,  ..., 2.0127e-04, 2.2186e-04,
          1.2442e-05],
         ...,
         [1.5387e-08, 6.4111e-08, 1.1089e-08,  ..., 1.7331e-07, 1.0000e+00,
   

In [35]:
# now we are going to add the predicitons of the vit models again becasue they are the best so that they have more effect
tta_preds += tta_preds[3:5]

In [36]:
# make toech.stack to make the tuple into tensor preds (5 models+ 2 extra vit)
tensor_tta_preds = torch.stack(tta_preds)
tensor_tta_preds.shape

torch.Size([7, 3469, 10])

In [37]:
# take the mean of preds
avg_preds = tensor_tta_preds.mean(0)
avg_preds.shape

torch.Size([3469, 10])

# Make the submission file

In [41]:
# make dls just to get the vocab from
dls = ImageDataLoaders.from_folder(trn_path)

In [44]:
dls.vocab

['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']

In [45]:
np.array(dls.vocab)

array(['bacterial_leaf_blight', 'bacterial_leaf_streak',
       'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart',
       'downy_mildew', 'hispa', 'normal', 'tungro'], dtype='<U24')

In [46]:
# make the arg max from the 10 classes
pred_idxs = avg_preds.argmax(dim=1)
#make an numpy array of the vocab
vocab = np.array(dls.vocab)
# read the sub file
ss = pd.read_csv(Path("../Data/plant_comp/sample_submission.csv"))
# make the label
ss['label'] = vocab[pred_idxs]
#make the csv
ss.to_csv('submission_3.csv', index=False)